# Семинар 2. Исследование методов линейной регрессии

#### Критерий оценивания:
#### Пункт 14: максимум 0,5 балла
#### Пункт 15: максимум 0,5 балла

#### Итого за работу: максимум 1 балл
#### P.S. пункты 14 и 15 без пунктов 1-13 не засчитываются, ответы на вопросы из ИИ не засчитываются

1. Загрузите датасет и выведите на экран первые несколько строк

In [1]:
# При необходимости: %pip install numpy pandas scikit-learn threadpoolctl
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
from threadpoolctl import threadpool_limits

DATA_PATH = Path("auto_dataset(3).csv")
if not DATA_PATH.exists():
    DATA_PATH = Path("auto_dataset.csv")
if not DATA_PATH.exists():
    raise FileNotFoundError("Put auto_dataset(3).csv next to this notebook.")

df = pd.read_csv(DATA_PATH)
if "price" not in df.columns:
    raise ValueError("The dataset must contain the price column.")
if df.isna().any().any():
    raise ValueError("The dataset contains missing values; inspect them first.")

print("Shape:", df.shape)
print("Missing values:", int(df.isna().sum().sum()))
print("Exact duplicate rows:", int(df.duplicated().sum()))
display(df.head())


Shape: (1000, 10)
Missing values: 0
Exact duplicate rows: 1


,brand,model,vehicleType,gearbox,fuelType,notRepairedDamage,powerPS,kilometer,autoAgeMonths,price
0,volkswagen,golf,kleinwagen,manuell,benzin,nein,75,150000,177,1500
1,skoda,fabia,kleinwagen,manuell,diesel,nein,69,90000,93,3600
2,bmw,3er,limousine,manuell,benzin,ja,102,150000,246,650
3,peugeot,2_reihe,cabrio,manuell,benzin,nein,109,150000,140,2200
4,mazda,3_reihe,limousine,manuell,benzin,nein,105,150000,136,2000


2. Разбейте выборку на признаки и ответы. Закодируйте категориальные признаки.

In [2]:
X_raw = df.drop(columns="price")
y_raw = df["price"].astype(float)

categorical_columns = X_raw.select_dtypes(include=["object", "category"]).columns.tolist()
numeric_columns = X_raw.select_dtypes(include="number").columns.tolist()

# Задаем правила кодирования; fit будет только на train в пункте 3.
preprocessor = ColumnTransformer([
    ("num", StandardScaler(), numeric_columns),
    ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical_columns),
])

print("Target: price")
print("Numeric columns:", numeric_columns)
print("Categorical columns:", categorical_columns)


Target: price
Numeric columns: ['powerPS', 'kilometer', 'autoAgeMonths']
Categorical columns: ['brand', 'model', 'vehicleType', 'gearbox', 'fuelType', 'notRepairedDamage']


3. Разбейте датасет на train val test в отношении 8:1:1

In [3]:
RANDOM_STATE = 0
X_train_raw, X_temp_raw, y_train, y_temp = train_test_split(
    X_raw, y_raw, test_size=0.2, random_state=RANDOM_STATE
)
X_val_raw, X_test_raw, y_val, y_test = train_test_split(
    X_temp_raw, y_temp, test_size=0.5, random_state=RANDOM_STATE
)

# Кодировщик и масштабы обучаются только по train.
X_train_encoded = preprocessor.fit_transform(X_train_raw)
X_val_encoded = preprocessor.transform(X_val_raw)
X_test_encoded = preprocessor.transform(X_test_raw)

# Столбец единиц добавляет свободный коэффициент.
X_train = np.column_stack([np.ones(len(X_train_raw)), X_train_encoded])
X_val = np.column_stack([np.ones(len(X_val_raw)), X_val_encoded])
X_test = np.column_stack([np.ones(len(X_test_raw)), X_test_encoded])

# Цены масштабируем для обучения, а прогнозы вернем в исходный масштаб.
y_mean = float(y_train.mean())
y_scale = float(y_train.std(ddof=0))
if not np.isfinite(y_scale) or y_scale <= 0:
    raise ValueError("Training prices must have positive finite variance.")
y_train_scaled = (y_train.to_numpy() - y_mean) / y_scale

split_labels = pd.Series(index=df.index, dtype="object")
for name, indices in [("train", X_train_raw.index), ("val", X_val_raw.index), ("test", X_test_raw.index)]:
    split_labels.loc[indices] = name

# Проверяем, что дубли не попали в разные выборки. Строки не удаляем.
duplicate_groups = df.groupby(df.columns.tolist(), dropna=False).indices
for indices in duplicate_groups.values():
    if len(indices) > 1 and split_labels.iloc[indices].nunique() != 1:
        raise ValueError("Identical rows ended up in different splits.")

assert (len(y_train), len(y_val), len(y_test)) == (800, 100, 100)
assert set(X_train_raw.index).isdisjoint(X_val_raw.index)
assert set(X_train_raw.index).isdisjoint(X_test_raw.index)
assert set(X_val_raw.index).isdisjoint(X_test_raw.index)

print("train / val / test:", len(y_train), len(y_val), len(y_test))
print("Features after encoding:", X_train_encoded.shape[1])
print("Weights including intercept:", X_train.shape[1])
feature_names = preprocessor.get_feature_names_out()
display(pd.DataFrame(X_train_encoded[:5], columns=feature_names))


train / val / test: 800 100 100
Features after encoding: 200
Weights including intercept: 201


,num__powerPS,num__kilometer,num__autoAgeMonths,cat__brand_alfa_romeo,cat__brand_audi,cat__brand_bmw,cat__brand_chevrolet,cat__brand_chrysler,cat__brand_citroen,cat__brand_dacia,cat__brand_daewoo,cat__brand_daihatsu,cat__brand_fiat,cat__brand_ford,cat__brand_honda,cat__brand_hyundai,cat__brand_jaguar,cat__brand_jeep,cat__brand_kia,cat__brand_lancia,cat__brand_land_rover,cat__brand_mazda,cat__brand_mercedes_benz,cat__brand_mini,cat__brand_mitsubishi,cat__brand_nissan,cat__brand_opel,cat__brand_peugeot,cat__brand_porsche,cat__brand_renault,cat__brand_rover,cat__brand_saab,cat__brand_seat,cat__brand_skoda,cat__brand_smart,cat__brand_subaru,cat__brand_suzuki,cat__brand_toyota,cat__brand_volkswagen,cat__brand_volvo,...,cat__model_sprinter,cat__model_stilo,cat__model_swift,cat__model_tiguan,cat__model_touareg,cat__model_touran,cat__model_transporter,cat__model_tt,cat__model_twingo,cat__model_up,cat__model_v40,cat__model_v50,cat__model_v_klasse,cat__model_vectra,cat__model_verso,cat__model_viano,cat__model_vito,cat__model_x_reihe,cat__model_yaris,cat__model_yeti,cat__model_ypsilon,cat__model_z_reihe,cat__model_zafira,cat__vehicleType_andere,cat__vehicleType_bus,cat__vehicleType_cabrio,cat__vehicleType_coupe,cat__vehicleType_kleinwagen,cat__vehicleType_kombi,cat__vehicleType_limousine,cat__vehicleType_suv,cat__gearbox_automatik,cat__gearbox_manuell,cat__fuelType_andere,cat__fuelType_benzin,cat__fuelType_diesel,cat__fuelType_hybrid,cat__fuelType_lpg,cat__notRepairedDamage_ja,cat__notRepairedDamage_nein
0,0.269143,0.656987,-0.270779,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0
1,-0.465162,-1.394095,-0.905544,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0
2,-0.399890,0.656987,-0.063509,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0
3,0.350733,0.656987,0.273305,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0
4,-1.183149,-0.624939,0.143761,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0


4. Исследуйте VGD с постоянным шагом n:

переберите n в логарифмической сетке от 10^-5 до 1

для каждого n:

* обучите VGD на train
* найдите и запомните R^2_train и Loss_train
* протестируйте VGD на val, запомните Loss_val

найдите наилучший n по минимальному Loss_val, запомните его Loss_train, R^2_train, Loss_val;

протестируйте VGD c лучшим n на test, запомните Loss_test, R^2_test, число итераций на test.

In [4]:
# Checked against Lecture 2 (uploaded PDF):
# TimeDecayLR, p. 1: eta(k) = lambda * (s0 / (s0 + k))**p;
# s0 = 1, p = 0.5, k starts at zero. Equivalent to lambda / sqrt(k + 1).
# MSE and full gradients: pp. 1-4; Momentum update: pp. 3-4; Adam: pp. 4-5.
# SGD uses a batch of one row: a special case of the batch formula on p. 2.
# Experiment choices retained from the previous notebook:
# - 11 grid values, 10,000 updates, stopping tolerance, seed and preprocessing;
# - zero initial weights and zero initial SAG gradient memory;
# - one iteration = one weight update, not one epoch;
# - stop when full training gradient infinity-norm <= TOL, or at MAX_ITER;
# - report Loss as normalized MSE: MSE / Var_train(price). Values are dimensionless and usually around 0.x.

STEP_GRID = np.logspace(-5, 0, 11)
MAX_ITER = 10_000
TOL = 1e-6
CHECK_EVERY = 100
DIVERGENCE_LOSS = 1e12  # MSE of standardized training prices
S0 = 1.0
POWER = 0.5  # TimeDecayLR parameters from Lecture 2, p. 1


def learning_rate(base_lr: float, iteration: int, decay: bool) -> float:
    """Lecture 2, p. 1: eta_k = lambda * (s0 / (s0 + k))**p."""
    if decay:
        return base_lr * (S0 / (S0 + iteration))**POWER
    return base_lr


def fit_optimizer(X: np.ndarray, y: np.ndarray, method: str,
                  base_lr: float, decay: bool = False,
                  max_iter: int = MAX_ITER, tol: float = TOL,
                  random_state: int = RANDOM_STATE) -> dict:
    """Fit only on the supplied training data. No validation/test data here."""
    allowed = {"VGD", "SGD", "SAG", "Momentum", "Adam"}
    if method not in allowed or base_lr <= 0 or max_iter < 1:
        raise ValueError("Check method, base_lr and max_iter.")
    X = np.asarray(X, dtype=float)
    y = np.asarray(y, dtype=float)
    if X.ndim != 2 or y.ndim != 1 or X.shape[0] != len(y):
        raise ValueError("X must be 2D and y must have one value per row.")
    if not np.isfinite(X).all() or not np.isfinite(y).all():
        raise ValueError("Training data must be finite.")

    n, d = X.shape
    w = np.zeros(d)
    velocity = np.zeros(d)
    first_moment = np.zeros(d)
    second_moment = np.zeros(d)
    momentum = 0.9
    beta1, beta2, adam_eps = 0.9, 0.999, 1e-8

    # Exact full MSE gradient: 2*(X.T @ (X @ w - y))/n = 2*(G @ w - c).
    # G and c are computed once, to avoid redundant matrix products.
    G = X.T @ X / n
    c = X.T @ y / n
    sample_indices = np.random.default_rng(random_state).integers(n, size=max_iter)
    if method == "SAG":
        gradient_memory = np.zeros((n, d))
        average_gradient = np.zeros(d)

    status = "max_iter"
    gradient_norm = float(np.linalg.norm(-2 * c, ord=np.inf))
    if gradient_norm <= tol:
        return {"weights": w, "iterations_train": 0,
                "status": "gradient_tol", "gradient_norm": gradient_norm}

    with np.errstate(over="ignore", invalid="ignore"):
        for k in range(max_iter):
            eta = learning_rate(base_lr, k, decay)
            if method in {"SGD", "SAG"}:
                i = sample_indices[k]
                g_i = 2 * (X[i] @ w - y[i]) * X[i]
                if method == "SGD":
                    gradient = g_i
                else:
                    average_gradient += (g_i - gradient_memory[i]) / n
                    gradient_memory[i] = g_i
                    gradient = average_gradient
            else:
                gradient = 2 * (G @ w - c)

            if method == "Momentum":
                # Lecture 2, pp. 3-4: h_{k+1} = 0.9*h_k + eta_k*gradient.
                # Here velocity stores h, including the learning-rate factor.
                velocity = momentum * velocity + eta * gradient
                w -= velocity
            elif method == "Adam":
                first_moment = beta1 * first_moment + (1 - beta1) * gradient
                second_moment = beta2 * second_moment + (1 - beta2) * gradient**2
                m_hat = first_moment / (1 - beta1**(k + 1))
                v_hat = second_moment / (1 - beta2**(k + 1))
                w -= eta * m_hat / (np.sqrt(v_hat) + adam_eps)
            else:
                w -= eta * gradient

            if not np.isfinite(w).all() or np.linalg.norm(w, ord=np.inf) > 1e12:
                status = "diverged"
                break

            if (k + 1) % CHECK_EVERY == 0 or k + 1 == max_iter:
                train_mse_scaled = float(np.mean((X @ w - y)**2))
                gradient_norm = float(np.linalg.norm(2 * (G @ w - c), ord=np.inf))
                if not np.isfinite(train_mse_scaled) or train_mse_scaled > DIVERGENCE_LOSS:
                    status = "diverged"
                    break
                if gradient_norm <= tol:
                    status = "gradient_tol"
                    break

    return {"weights": w, "iterations_train": k + 1,
            "status": status, "gradient_norm": gradient_norm}


def predict_price(X: np.ndarray, weights: np.ndarray) -> np.ndarray:
    return (X @ weights) * y_scale + y_mean


def price_metrics(y_true, y_pred) -> dict:
    # Normalized MSE: divide ordinary MSE by the variance of price on train.
    # Since y_scale = std(train price), y_scale**2 = Var_train(price).
    # This keeps the same ordering of models but makes Loss dimensionless (typically 0.xx).
    loss_normalized = float(mean_squared_error(y_true, y_pred) / (y_scale ** 2))
    return {"Loss": loss_normalized,
            "R2": float(r2_score(y_true, y_pred))}


experiments = {}


def run_experiment(method: str, decay: bool) -> dict:
    trials = []
    fitted = []
    # Small matrices are faster with a single BLAS thread.
    with threadpool_limits(limits=1, user_api="blas"):
        for param in STEP_GRID:
            model = fit_optimizer(X_train, y_train_scaled, method, float(param), decay)
            fitted.append(model)
            if model["status"] == "diverged":
                loss_train, r2_train, loss_val = np.inf, -np.inf, np.inf
            else:
                train_metrics = price_metrics(y_train, predict_price(X_train, model["weights"]))
                loss_train, r2_train = train_metrics["Loss"], train_metrics["R2"]
                loss_val = price_metrics(y_val, predict_price(X_val, model["weights"]))["Loss"]
            trials.append({"parameter": float(param), "Loss_train": loss_train,
                           "R2_train": r2_train, "Loss_val": loss_val,
                           "iterations_train": model["iterations_train"],
                           "status": model["status"]})

        trials = pd.DataFrame(trials)
        finite = np.isfinite(trials["Loss_val"])
        if not finite.any():
            raise RuntimeError("All candidates diverged; inspect data and step sizes.")
        best_index = trials.loc[finite, "Loss_val"].idxmin()
        best = fitted[int(best_index)]
        best_param = float(trials.loc[best_index, "parameter"])

        # Test is evaluated only AFTER selecting the step on validation.
        test_metrics = price_metrics(y_test, predict_price(X_test, best["weights"]))
        row = {"method": method, "schedule": "TimeDecayLR" if decay else "constant",
               "best_parameter": best_param,
               "best_step": (f"{best_param:.8g}*({S0:g}/({S0:g}+k))**{POWER:g}"
                             if decay else f"{best_param:.8g}"),
               "Loss_train": float(trials.loc[best_index, "Loss_train"]),
               "Loss_val": float(trials.loc[best_index, "Loss_val"]),
               "Loss_test": test_metrics["Loss"],
               "R2_train": float(trials.loc[best_index, "R2_train"]),
               "R2_test": test_metrics["R2"],
               "iterations_train": best["iterations_train"],
               "weight_updates_test": 0,
               "status": best["status"]}

    result = {"summary": row, "trials": trials, "model": best}
    experiments[(method, decay)] = result
    print(method, "/", row["schedule"])
    display(trials.round({"parameter": 8, "Loss_train": 6, "Loss_val": 6, "R2_train": 6}))
    print("Best candidate selected ONLY by minimum Loss_val:")
    display(pd.DataFrame([row]).round({"Loss_train": 6, "Loss_val": 6, "Loss_test": 6,
                                     "R2_train": 6, "R2_test": 6}))
    return result


result_vgd_constant = run_experiment("VGD", decay=False)


VGD / constant


,parameter,Loss_train,R2_train,Loss_val,iterations_train,status
0,0.000010,0.726166,0.273834,0.902992,10000,max_iter
1,0.000032,0.474031,0.525969,0.567221,10000,max_iter
2,0.000100,0.356654,0.643346,0.391495,10000,max_iter
3,0.000316,0.330294,0.669706,0.344739,10000,max_iter
4,0.001000,0.312689,0.687311,0.316718,10000,max_iter
5,0.003162,0.288155,0.711845,0.282776,10000,max_iter
6,0.010000,0.255911,0.744089,0.247729,10000,max_iter
7,0.031623,0.229673,0.770327,0.252299,10000,max_iter
8,0.100000,0.217257,0.782743,0.289120,10000,max_iter
9,0.316228,inf,-inf,inf,200,diverged


Best candidate selected ONLY by minimum Loss_val:


,method,schedule,best_parameter,best_step,Loss_train,Loss_val,Loss_test,R2_train,R2_test,iterations_train,weight_updates_test,status
0,VGD,constant,0.01,0.01,0.255911,0.247729,0.160928,0.744089,0.759261,10000,0,max_iter


5. Исследуйте VGD с переменным шагом n(lyamda) по формуле TimeDecayLR (из лекции):

переберите lyamda в логарифмической сетке от 10^-5 до 1

для каждого lyamda:

* обучите VGD на train
* найдите и запомните R^2_train и Loss_train
* протестируйте VGD на val, запомните Loss_val

найдите наилучший lyamda по минимальному Loss_val, запомните его Loss_train, R^2_train, Loss_val;

протестируйте VGD c n(best_lyamda) на test, запомните Loss_test, R^2_test, число итераций на test.

In [5]:
result_vgd_decay = run_experiment("VGD", decay=True)


VGD / TimeDecayLR


,parameter,Loss_train,R2_train,Loss_val,iterations_train,status
0,0.000010,0.992872,0.007128,1.243648,10000,max_iter
1,0.000032,0.977739,0.022261,1.224553,10000,max_iter
2,0.000100,0.932306,0.067694,1.167077,10000,max_iter
3,0.000316,0.810115,0.189885,1.011276,10000,max_iter
4,0.001000,0.573698,0.426302,0.702613,10000,max_iter
5,0.003162,0.381421,0.618579,0.432698,10000,max_iter
6,0.010000,0.339155,0.660845,0.360175,10000,max_iter
7,0.031623,0.319790,0.680210,0.327390,10000,max_iter
8,0.100000,0.299450,0.700550,0.298048,10000,max_iter
9,0.316228,0.269166,0.730834,0.259602,10000,max_iter


Best candidate selected ONLY by minimum Loss_val:


,method,schedule,best_parameter,best_step,Loss_train,Loss_val,Loss_test,R2_train,R2_test,iterations_train,weight_updates_test,status
0,VGD,TimeDecayLR,1.0,1*(1/(1+k))**0.5,0.238502,0.244022,0.160006,0.761498,0.760641,10000,0,max_iter


6. Исследуйте SGD с постоянным шагом n:

переберите n в логарифмической сетке от 10^-5 до 1

для каждого n:

* обучите SGD на train
* найдите и запомните R^2_train и Loss_train
* протестируйте SGD на val, запомните Loss_val

найдите наилучший n по минимальному Loss_val, запомните его Loss_train, R^2_train, Loss_val;

протестируйте SGD c лучшим n на test, запомните Loss_test, R^2_test, число итераций на test.

In [6]:
result_sgd_constant = run_experiment("SGD", decay=False)


SGD / constant


,parameter,Loss_train,R2_train,Loss_val,iterations_train,status
0,0.000010,0.724873,0.275127,0.900363,10000,max_iter
1,0.000032,0.472425,0.527575,0.565224,10000,max_iter
2,0.000100,0.356183,0.643817,0.395201,10000,max_iter
3,0.000316,0.333996,0.666004,0.358799,10000,max_iter
4,0.001000,0.329267,0.670733,0.353407,10000,max_iter
5,0.003162,0.366208,0.633792,0.394975,10000,max_iter
6,0.010000,0.617467,0.382533,0.667229,10000,max_iter
7,0.031623,2.485507,-1.485507,2.478374,10000,max_iter
8,0.100000,inf,-inf,inf,1700,diverged
9,0.316228,inf,-inf,inf,45,diverged


Best candidate selected ONLY by minimum Loss_val:


,method,schedule,best_parameter,best_step,Loss_train,Loss_val,Loss_test,R2_train,R2_test,iterations_train,weight_updates_test,status
0,SGD,constant,0.001,0.001,0.329267,0.353407,0.206042,0.670733,0.691774,10000,0,max_iter


7. Исследуйте SGD с переменным шагом n(lyamda) по формуле TimeDecayLR (из лекции):

переберите lyamda в логарифмической сетке от 10^-5 до 1

для каждого lyamda:

* обучите SGD на train
* найдите и запомните R^2_train и Loss_train
* протестируйте SGD на val, запомните Loss_val

найдите наилучший lyamda по минимальному Loss_val, запомните его Loss_train, R^2_train, Loss_val;

протестируйте SGD c n(best_lyamda) на test, запомните Loss_test, R^2_test, число итераций на test.

In [7]:
result_sgd_decay = run_experiment("SGD", decay=True)


SGD / TimeDecayLR


,parameter,Loss_train,R2_train,Loss_val,iterations_train,status
0,0.000010,0.992540,0.007460,1.243154,10000,max_iter
1,0.000032,0.976712,0.023288,1.223033,10000,max_iter
2,0.000100,0.929281,0.070719,1.162659,10000,max_iter
3,0.000316,0.802487,0.197513,1.000548,10000,max_iter
4,0.001000,0.561866,0.438134,0.687667,10000,max_iter
5,0.003162,0.376270,0.623730,0.429613,10000,max_iter
6,0.010000,0.338457,0.661543,0.366090,10000,max_iter
7,0.031623,0.323961,0.676039,0.344762,10000,max_iter
8,0.100000,0.316211,0.683789,0.335062,10000,max_iter
9,0.316228,0.346088,0.653912,0.341046,10000,max_iter


Best candidate selected ONLY by minimum Loss_val:


,method,schedule,best_parameter,best_step,Loss_train,Loss_val,Loss_test,R2_train,R2_test,iterations_train,weight_updates_test,status
0,SGD,TimeDecayLR,0.1,0.1*(1/(1+k))**0.5,0.316211,0.335062,0.199136,0.683789,0.702105,10000,0,max_iter


8. Исследуйте SAG с постоянным шагом n:

переберите n в логарифмической сетке от 10^-5 до 1

для каждого n:

* обучите SAG на train
* найдите и запомните R^2_train и Loss_train
* протестируйте SAG на val, запомните Loss_val

найдите наилучший n по минимальному Loss_val, запомните его Loss_train, R^2_train, Loss_val;

протестируйте SAG c лучшим n на test, запомните Loss_test, R^2_test, число итераций на test.

In [8]:
result_sag_constant = run_experiment("SAG", decay=False)


SAG / constant


,parameter,Loss_train,R2_train,Loss_val,iterations_train,status
0,0.000010,0.736251,0.263749,0.916068,10000,max_iter
1,0.000032,0.474671,0.525329,0.568500,10000,max_iter
2,0.000100,0.355678,0.644322,0.388905,10000,max_iter
3,0.000316,0.330434,0.669566,0.343476,10000,max_iter
4,0.001000,0.313706,0.686294,0.318032,10000,max_iter
5,0.003162,0.289573,0.710427,0.284460,10000,max_iter
6,0.010000,0.255216,0.744784,0.248978,10000,max_iter
7,0.031623,0.246968,0.753032,0.276479,10000,max_iter
8,0.100000,1286.760016,-1285.760016,1516.015544,10000,max_iter
9,0.316228,inf,-inf,inf,7200,diverged


Best candidate selected ONLY by minimum Loss_val:


,method,schedule,best_parameter,best_step,Loss_train,Loss_val,Loss_test,R2_train,R2_test,iterations_train,weight_updates_test,status
0,SAG,constant,0.01,0.01,0.255216,0.248978,0.16185,0.744784,0.757882,10000,0,max_iter


9. Исследуйте SAG с переменным шагом n(lyamda) по формуле TimeDecayLR (из лекции):

переберите lyamda в логарифмической сетке от 10^-5 до 1

для каждого lyamda:

* обучите SAG на train
* найдите и запомните R^2_train и Loss_train
* протестируйте SAG на val, запомните Loss_val

найдите наилучший lyamda по минимальному Loss_val, запомните его Loss_train, R^2_train, Loss_val;

протестируйте SAG c n(best_lyamda) на test, запомните Loss_test, R^2_test, число итераций на test.

In [9]:
result_sag_decay = run_experiment("SAG", decay=True)


SAG / TimeDecayLR


,parameter,Loss_train,R2_train,Loss_val,iterations_train,status
0,0.000010,0.994468,0.005532,1.245644,10000,max_iter
1,0.000032,0.982661,0.017339,1.230720,10000,max_iter
2,0.000100,0.946694,0.053306,1.185182,10000,max_iter
3,0.000316,0.845597,0.154403,1.056501,10000,max_iter
4,0.001000,0.624463,0.375537,0.770131,10000,max_iter
5,0.003162,0.394602,0.605398,0.453782,10000,max_iter
6,0.010000,0.343315,0.656685,0.366696,10000,max_iter
7,0.031623,0.322903,0.677097,0.332859,10000,max_iter
8,0.100000,0.304580,0.695420,0.306558,10000,max_iter
9,0.316228,0.276043,0.723957,0.268104,10000,max_iter


Best candidate selected ONLY by minimum Loss_val:


,method,schedule,best_parameter,best_step,Loss_train,Loss_val,Loss_test,R2_train,R2_test,iterations_train,weight_updates_test,status
0,SAG,TimeDecayLR,1.0,1*(1/(1+k))**0.5,0.2461,0.246604,0.158173,0.7539,0.763382,10000,0,max_iter


10. Исследуйте Momentum с постоянным шагом n:

переберите n в логарифмической сетке от 10^-5 до 1

для каждого n:

* обучите Momentum на train
* найдите и запомните R^2_train и Loss_train
* протестируйте Momentum на val, запомните Loss_val

найдите наилучший n по минимальному Loss_val, запомните его Loss_train, R^2_train, Loss_val;

протестируйте Momentum c лучшим n на test, запомните Loss_test, R^2_test, число итераций на test.

In [10]:
result_momentum_constant = run_experiment("Momentum", decay=False)


Momentum / constant


,parameter,Loss_train,R2_train,Loss_val,iterations_train,status
0,0.000010,0.356646,0.643354,0.391474,10000,max_iter
1,0.000032,0.330298,0.669702,0.344737,10000,max_iter
2,0.000100,0.312700,0.687300,0.316729,10000,max_iter
3,0.000316,0.288169,0.711831,0.282790,10000,max_iter
4,0.001000,0.255922,0.744078,0.247719,10000,max_iter
5,0.003162,0.229674,0.770326,0.252292,10000,max_iter
6,0.010000,0.217259,0.782741,0.289131,10000,max_iter
7,0.031623,0.211812,0.788188,0.325390,10000,max_iter
8,0.100000,0.211169,0.788831,0.349795,10000,max_iter
9,0.316228,0.211147,0.788853,0.358684,9500,gradient_tol


Best candidate selected ONLY by minimum Loss_val:


,method,schedule,best_parameter,best_step,Loss_train,Loss_val,Loss_test,R2_train,R2_test,iterations_train,weight_updates_test,status
0,Momentum,constant,0.001,0.001,0.255922,0.247719,0.160937,0.744078,0.759248,10000,0,max_iter


11. Исследуйте Momentum с переменным шагом n(lyamda) по формуле TimeDecayLR (из лекции):

переберите lyamda в логарифмической сетке от 10^-5 до 1

для каждого lyamda:

* обучите Momentum на train
* найдите и запомните R^2_train и Loss_train
* протестируйте Momentum на val, запомните Loss_val

найдите наилучший lyamda по минимальному Loss_val, запомните его Loss_train, R^2_train, Loss_val;

протестируйте Momentum c n(best_lyamda) на test, запомните Loss_test, R^2_test, число итераций на test.

In [11]:
result_momentum_decay = run_experiment("Momentum", decay=True)


Momentum / TimeDecayLR


,parameter,Loss_train,R2_train,Loss_val,iterations_train,status
0,0.000010,0.932328,0.067672,1.167105,10000,max_iter
1,0.000032,0.810134,0.189866,1.011301,10000,max_iter
2,0.000100,0.573564,0.426436,0.702438,10000,max_iter
3,0.000316,0.381249,0.618751,0.432426,10000,max_iter
4,0.001000,0.339138,0.660862,0.360123,10000,max_iter
5,0.003162,0.319784,0.680216,0.327365,10000,max_iter
6,0.010000,0.299447,0.700553,0.298036,10000,max_iter
7,0.031623,0.269153,0.730847,0.259562,10000,max_iter
8,0.100000,0.238480,0.761520,0.244001,10000,max_iter
9,0.316228,0.221167,0.778833,0.273695,10000,max_iter


Best candidate selected ONLY by minimum Loss_val:


,method,schedule,best_parameter,best_step,Loss_train,Loss_val,Loss_test,R2_train,R2_test,iterations_train,weight_updates_test,status
0,Momentum,TimeDecayLR,0.1,0.1*(1/(1+k))**0.5,0.23848,0.244001,0.160018,0.76152,0.760623,10000,0,max_iter


12. Исследуйте Adam с постоянным шагом n:

переберите n в логарифмической сетке от 10^-5 до 1

для каждого n:

* обучите Adam на train
* найдите и запомните R^2_train и Loss_train
* протестируйте Adam на val, запомните Loss_val

найдите наилучший n по минимальному Loss_val, запомните его Loss_train, R^2_train, Loss_val;

протестируйте Adam c лучшим n на test, запомните Loss_test, R^2_test, число итераций на test.

In [12]:
result_adam_constant = run_experiment("Adam", decay=False)


Adam / constant


,parameter,Loss_train,R2_train,Loss_val,iterations_train,status
0,0.000010,0.549237,0.450763,0.737912,10000,max_iter
1,0.000032,0.293897,0.706103,0.335531,10000,max_iter
2,0.000100,0.227323,0.772677,0.267375,10000,max_iter
3,0.000316,0.211487,0.788513,0.348586,10000,max_iter
4,0.001000,0.211147,0.788853,0.365350,8400,gradient_tol
5,0.003162,0.211147,0.788853,0.365042,5000,gradient_tol
6,0.010000,0.211147,0.788853,0.363608,2400,gradient_tol
7,0.031623,0.211147,0.788853,0.361502,1100,gradient_tol
8,0.100000,0.211147,0.788853,0.360364,1400,gradient_tol
9,0.316228,0.211147,0.788853,0.359320,400,gradient_tol


Best candidate selected ONLY by minimum Loss_val:


,method,schedule,best_parameter,best_step,Loss_train,Loss_val,Loss_test,R2_train,R2_test,iterations_train,weight_updates_test,status
0,Adam,constant,0.0001,0.0001,0.227323,0.267375,0.168033,0.772677,0.748633,10000,0,max_iter


13. Исследуйте Adam с переменным шагом n(lyamda) по формуле TimeDecayLR (из лекции):

переберите lyamda в логарифмической сетке от 10^-5 до 1

для каждого lyamda:

* обучите Adam на train
* найдите и запомните R^2_train и Loss_train
* протестируйте Adam на val, запомните Loss_val

найдите наилучший lyamda по минимальному Loss_val, запомните его Loss_train, R^2_train, Loss_val;

протестируйте Adam c n(best_lyamda) на test, запомните Loss_test, R^2_test, число итераций на test.

In [13]:
result_adam_decay = run_experiment("Adam", decay=True)


Adam / TimeDecayLR


,parameter,Loss_train,R2_train,Loss_val,iterations_train,status
0,0.000010,0.987522,0.012478,1.239327,10000,max_iter
1,0.000032,0.961066,0.038934,1.211098,10000,max_iter
2,0.000100,0.882008,0.117992,1.125788,10000,max_iter
3,0.000316,0.675810,0.324190,0.893570,10000,max_iter
4,0.001000,0.366465,0.633535,0.471185,10000,max_iter
5,0.003162,0.240449,0.759551,0.252033,10000,max_iter
6,0.010000,0.213757,0.786243,0.325245,10000,max_iter
7,0.031623,0.211147,0.788853,0.363870,7800,gradient_tol
8,0.100000,0.211147,0.788853,0.361337,3700,gradient_tol
9,0.316228,0.211147,0.788853,0.359879,2800,gradient_tol


Best candidate selected ONLY by minimum Loss_val:


,method,schedule,best_parameter,best_step,Loss_train,Loss_val,Loss_test,R2_train,R2_test,iterations_train,weight_updates_test,status
0,Adam,TimeDecayLR,0.003162,0.0031622777*(1/(1+k))**0.5,0.240449,0.252033,0.164666,0.759551,0.75367,10000,0,max_iter


14. Постройте итоговую сравнительную таблицу со следующими столбцами:

1) название метода

2) значение лучшего шага (если n) или функция лучшего шага (если n(lyamda))

3) Loss_train

4) Loss_test

5) R^2 train

6) R^2 test

7) число итераций на test

In [14]:
results = pd.DataFrame([item["summary"] for item in experiments.values()])
assert len(results) == 10

# Итерации модели для test = iterations_train. На самом test веса не меняются.
columns = ["method", "schedule", "best_step", "Loss_train", "Loss_val", "Loss_test",
           "R2_train", "R2_test", "iterations_train", "weight_updates_test", "status"]
print("Loss = normalized MSE = MSE / Var_train(price)")
display(results[columns].round({"Loss_train": 6, "Loss_val": 6, "Loss_test": 6,
                               "R2_train": 6, "R2_test": 6}))
results.to_csv("sem_2_results_normalized.csv", index=False, encoding="utf-8-sig")
print("Saved: sem_2_results_normalized.csv")


Loss = normalized MSE = MSE / Var_train(price)


,method,schedule,best_step,Loss_train,Loss_val,Loss_test,R2_train,R2_test,iterations_train,weight_updates_test,status
0,VGD,constant,0.01,0.255911,0.247729,0.160928,0.744089,0.759261,10000,0,max_iter
1,VGD,TimeDecayLR,1*(1/(1+k))**0.5,0.238502,0.244022,0.160006,0.761498,0.760641,10000,0,max_iter
2,SGD,constant,0.001,0.329267,0.353407,0.206042,0.670733,0.691774,10000,0,max_iter
3,SGD,TimeDecayLR,0.1*(1/(1+k))**0.5,0.316211,0.335062,0.199136,0.683789,0.702105,10000,0,max_iter
4,SAG,constant,0.01,0.255216,0.248978,0.161850,0.744784,0.757882,10000,0,max_iter
5,SAG,TimeDecayLR,1*(1/(1+k))**0.5,0.246100,0.246604,0.158173,0.753900,0.763382,10000,0,max_iter
6,Momentum,constant,0.001,0.255922,0.247719,0.160937,0.744078,0.759248,10000,0,max_iter
7,Momentum,TimeDecayLR,0.1*(1/(1+k))**0.5,0.238480,0.244001,0.160018,0.761520,0.760623,10000,0,max_iter
8,Adam,constant,0.0001,0.227323,0.267375,0.168033,0.772677,0.748633,10000,0,max_iter
9,Adam,TimeDecayLR,0.0031622777*(1/(1+k))**0.5,0.240449,0.252033,0.164666,0.759551,0.753670,10000,0,max_iter


Saved: sem_2_results_normalized.csv


15. Сделайте вывод о том, какой метод и шаг линейной регрессии самый лучший для данной выборки и ответьте на вопросы:

1) почему именно этот метод и этот шаг самый лучший (по каким данным из таблицы вы сделали такой вывод)

2) расскажите простыми словами суть R^2_train и R^2_test?

3) как R^2_train и R^2_test помогают сравнивать методы? почему оба эти значения надо вычислять для данного выбора?

1) Лучший по результатам тестовой выборки оказался SAG с TimeDecayLR и параметром лямбда=1. У этого варианта самый маленький Loss_test и самый большой R^2_test, это значит что SAG лучше предсказывает цены. При этом R^2_train очень схож с R^2_test, и поэтому не должно быть большого разрыва между качеством на обучающих и тестовый данных.

2) R^2_train показывает, насколько хорошо модель объясняет цены, основываясь на тех данных, на которых она уже обучалась.

    R^2_test показывает, насколько хорошо модель работает с новыми автомобилями, которые она не встречала во время своего обучения.
(чем ближе значение к 1, тем лучше модель описывает данные)

3) Оба нужны для оценки модели с разных "сторон".
Оба значения важны, поскольку после их вычисления можно сравнить значения - нередко R^2_train гораздо больше R^2_test, это означает, что модель черезчур основывается на обучающую выборку.